# Langextract Functions Review

## `langextract.extract()` review

In [ ]:
%load_ext rich

In [ ]:
import warnings

import langextract as lx

from langextract import annotation
from langextract import data
from langextract import factory
from langextract import prompting
from langextract import resolver

In [ ]:
debug = True
max_workers = 16
batch_length = 16

In [ ]:
prompt_description = """
    Sos un asistente especializado en el análisis de documentos judiciales.
    Tu tarea es identificar y extraer menciones de información sensible para su posterior anonimización.
    Debés detectar fragmentos textuales que correspondan a cualquiera de las siguientes entidades:

    - "BANCO": Nombre de una entidad bancaria, pública o privada.
    - "CBU": Código Bancario Uniforme (22 dígitos) de una cuenta.
    - "CORREO_ELECTRONICO": Dirección de correo electrónico.
    - "CUIJ": Código Único de Identificación Jurídica de causas judiciales.
    - "CUIT_CUIL": Número de CUIT o CUIL de una persona física o jurídica.
    - "DIRECCION": Dirección postal específica (calle, número, etc.).
    - "DNI": Número de Documento Nacional de Identidad u otro documento identificatorio.
    - "EDAD": Edad explícita de una persona.
    - "ESTUDIOS": Nivel o institución educativa que permita identificar a la persona (ej. "primario incompleto", "secundario completo", "Licenciado en…").
    - "FECHA": Fecha completa o parcial (día, mes y/o año).
    - "LINK": Enlace o URL a una página web.
    - "LOC": Localización geográfica específica (ciudad, barrio, comisaría, etc.).
    - "MARCA_AUTOMOVIL": Marca de un vehículo (ej. Toyota, Ford).
    - "NACIONALIDAD": Nacionalidad de una persona (ej. "argentino", "brasileña").
    - "NUM_ACTUACION": Número identificatorio de una actuación administrativa o contravencional.
    - "NUM_CAJA_AHORRO": Número completo de una caja de ahorro o cuenta bancaria.
    - "NUM_EXPEDIENTE": Número de expediente judicial o administrativo.
    - "NUM_MATRICULA": Número de matrícula profesional o académica.
    - "PATENTE_DOMINIO": Patente o dominio de un vehículo.
    - "PER": Nombre y apellido(s) de una persona física. Los nombres inicializados y los apodos también cuentan como información sensible a anonimizar.
    - "TELEFONO": Número telefónico (fijo o celular).
"""

In [ ]:
examples = [
    lx.data.ExampleData(
        text="El 5 de mayo de 2023 el señor Fiscal indicó que realizó distintas medidas de prueba y que del resultado surge que tanto la investigada como el menor Juan Pérez se domicilian en la calle Sarmiento 1234, de la localidad de Moreno, por lo que solicitó que se declare la incompetencia en razón del territorio y se envíe el caso al Juzgado de Garantías que corresponda del Departamento Judicial de Moreno, con jurisdicción en el partido de Moreno.",
        extractions=[
            lx.data.Extraction(
                extraction_class="FECHA", extraction_text="5 de mayo de 2023"
            ),
            lx.data.Extraction(
                extraction_class="PER", extraction_text="Juan Pérez"
            ),
            lx.data.Extraction(
                extraction_class="DIRECCION", extraction_text="Sarmiento 1234"
            ),
            lx.data.Extraction(
                extraction_class="LOC", extraction_text="Moreno"
            ),
        ],
    ),
    lx.data.ExampleData(
        text="JUZGADO DE 1RA INSTANCIA EN LO PENAL CONTRAVENCIONAL Y DE FALTAS N°10 SECRETARIA N°19\nCarlos Gómez sobre 84 - HOMICIDIO CULPOSO Y OTROS\nNúmero: 52345/2022\nCUIJ: 12-34567890-1\nActuación Nro: 2022-009876",
        extractions=[
            lx.data.Extraction(
                extraction_class="PER", extraction_text="Carlos Gómez"
            ),
            lx.data.Extraction(
                extraction_class="NUM_EXPEDIENTE", extraction_text="52345/2022"
            ),
            lx.data.Extraction(
                extraction_class="CUIJ", extraction_text="12-34567890-1"
            ),
            lx.data.Extraction(
                extraction_class="NUM_ACTUACION", extraction_text="2022-009876"
            ),
        ],
    ),
    lx.data.ExampleData(
        text="Acusado: Miguel Torres, DNI 30123456, nacido el 14/02/1990, de 34 años de edad, de nacionalidad paraguaya, género varón cis, con estudios secundarios completos, hizo hasta 3er año porque fue padre joven, con último domicilio en Av. Corrientes 3456, de esta ciudad, donde vive con su hermana y su cuñado Jorge Pérez. Tiene dos hijos a su cargo, de 5 y 8 años. Su hijo de 8 vive con él, su hija de 5 vive con su madre, Laura Fernández.",
        extractions=[
            lx.data.Extraction(
                extraction_class="PER", extraction_text="Miguel Torres"
            ),
            lx.data.Extraction(
                extraction_class="DNI", extraction_text="30123456"
            ),
            lx.data.Extraction(
                extraction_class="FECHA", extraction_text="14/02/1990"
            ),
            lx.data.Extraction(extraction_class="EDAD", extraction_text="34"),
            lx.data.Extraction(
                extraction_class="NACIONALIDAD", extraction_text="paraguaya"
            ),
            lx.data.Extraction(
                extraction_class="ESTUDIOS",
                extraction_text="estudios secundarios completos",
            ),
            lx.data.Extraction(
                extraction_class="DIRECCION",
                extraction_text="Av. Corrientes 3456",
            ),
            lx.data.Extraction(
                extraction_class="PER", extraction_text="Jorge Pérez"
            ),
            lx.data.Extraction(extraction_class="EDAD", extraction_text="5"),
            lx.data.Extraction(extraction_class="EDAD", extraction_text="8"),
            lx.data.Extraction(
                extraction_class="PER", extraction_text="Laura Fernández"
            ),
        ],
    ),
    lx.data.ExampleData(
        text="El testigo Juan López dejó asentado su número de contacto: 11-2345-6789. Indicó que la médica Dra. Ana García, MN 12345, asistió al lugar donde se hallaba un vehículo Volkswagen, patente AB123CD.",
        extractions=[
            lx.data.Extraction(
                extraction_class="PER", extraction_text="Juan López"
            ),
            lx.data.Extraction(
                extraction_class="TELEFONO", extraction_text="11-2345-6789"
            ),
            lx.data.Extraction(
                extraction_class="PER", extraction_text="Ana García"
            ),
            lx.data.Extraction(
                extraction_class="NUM_MATRICULA", extraction_text="12345"
            ),
            lx.data.Extraction(
                extraction_class="MARCA_AUTOMOVIL",
                extraction_text="Volkswagen",
            ),
            lx.data.Extraction(
                extraction_class="PATENTE_DOMINIO", extraction_text="AB123CD"
            ),
        ],
    ),
    lx.data.ExampleData(
        text="Se identificó una transferencia bancaria con los siguientes datos: CUIT 20-12345678-3, CBU 2850590940090412345671, Caja de Ahorro N° 12345678, Banco Nación.",
        extractions=[
            lx.data.Extraction(
                extraction_class="CUIT_CUIL", extraction_text="20-12345678-3"
            ),
            lx.data.Extraction(
                extraction_class="CBU",
                extraction_text="2850590940090412345671",
            ),
            lx.data.Extraction(
                extraction_class="NUM_CAJA_AHORRO", extraction_text="12345678"
            ),
            lx.data.Extraction(
                extraction_class="BANCO", extraction_text="Banco Nación"
            ),
        ],
    ),
    lx.data.ExampleData(
        text="Para mayor información, comunicarse a fiscalia.central@justicia.gob.ar o visitar el sitio https://justicia.gob.ar/actuaciones.",
        extractions=[
            lx.data.Extraction(
                extraction_class="CORREO_ELECTRONICO",
                extraction_text="fiscalia.central@justicia.gob.ar",
            ),
            lx.data.Extraction(
                extraction_class="LINK",
                extraction_text="https://justicia.gob.ar/actuaciones",
            ),
        ],
    ),
]

In [ ]:
# Examples assertion
if not examples:
    raise ValueError(
        "Examples are required for reliable extraction. Please provide at least"
        " one ExampleData object with sample extractions."
    )

In [ ]:
# Debug and max_workers configuration
if debug:
    # pylint: disable=import-outside-toplevel
    from langextract import debug_utils

    debug_utils.configure_debug_logging()

if max_workers is not None and batch_length < max_workers:
    warnings.warn(
        f"batch_length ({batch_length}) < max_workers ({max_workers}). "
        f"Only {batch_length} workers will be used. "
        "Set batch_length >= max_workers for optimal parallelization.",
        UserWarning,
    )

In [ ]:
# URL handling (uncomment if needed)
# if isinstance(text_or_documents, str) and io.is_url(text_or_documents):
#    text_or_documents = io.download_text_from_url(text_or_documents)

In [ ]:
# ===>  Prompt template
prompt_template = prompting.PromptTemplateStructured(
    description=prompt_description
)
prompt_template

In [ ]:
prompt_template.examples.extend(examples)
prompt_template

In [ ]:
print(
    prompting.QAPromptGenerator(
        prompt_template,
        format_type=data.FormatType.JSON,
        examples_heading="Ejemplos:\n--------\n",
        question_prefix="Input: ",
        answer_prefix="Output: ",
        fence_output=False,
    ).render("Hola")
)

In [ ]:
# # Model
# language_model = None

# if model:
#     language_model = model
#     if fence_output is not None:
#     language_model.set_fence_output(fence_output)
#     if use_schema_constraints:
#     warnings.warn(
#         "'use_schema_constraints' is ignored when 'model' is provided. "
#         "The model should already be configured with schema constraints.",
#         UserWarning,
#         stacklevel=2,
#     )
# elif config:
#     if use_schema_constraints:
#     warnings.warn(
#         "With 'config', schema constraints are still applied via examples. "
#         "Or pass explicit schema in config.provider_kwargs.",
#         UserWarning,
#         stacklevel=2,
#     )

#     language_model = factory.create_model(
#         config=config,
#         examples=prompt_template.examples if use_schema_constraints else None,
#         use_schema_constraints=use_schema_constraints,
#         fence_output=fence_output,
#     )
# else:
#     if language_model_type != inference.GeminiLanguageModel:
#     warnings.warn(
#         "'language_model_type' is deprecated and will be removed in v2.0.0. "
#         "Use model, config, or model_id parameters instead.",
#         DeprecationWarning,
#         stacklevel=2,
#     )

#     base_lm_kwargs: dict[str, Any] = {
#         "api_key": api_key,
#         "format_type": format_type,
#         "temperature": temperature,
#         "model_url": model_url,
#         "base_url": model_url,
#         "max_workers": max_workers,
#     }

#     # TODO(v2.0.0): Remove gemini_schema parameter
#     if "gemini_schema" in (language_model_params or {}):
#     warnings.warn(
#         "'gemini_schema' is deprecated. Schema constraints are now "
#         "automatically handled. This parameter will be ignored.",
#         DeprecationWarning,
#         stacklevel=2,
#     )
#     language_model_params = dict(language_model_params or {})
#     language_model_params.pop("gemini_schema", None)

#     base_lm_kwargs.update(language_model_params or {})
#     filtered_kwargs = {k: v for k, v in base_lm_kwargs.items() if v is not None}
#     config = factory.ModelConfig(
#         model_id=model_id, provider_kwargs=filtered_kwargs
#     )

#     language_model = factory.create_model(
#         config=config,
#         examples=prompt_template.examples if use_schema_constraints else None,
#         use_schema_constraints=use_schema_constraints,
#         fence_output=fence_output,
#     )

# fence_output = language_model.requires_fence_output

In [ ]:
model_id = "deepseek-r1:14b"
language_model_params = {"num_ctx": 12288}

use_schema_constraints = True  # False
fence_output = False
format_type = data.FormatType.JSON

base_lm_kwargs = {
    "api_key": None,
    "format_type": format_type,
    "temperature": 0,
    "model_url": "http://localhost:11434",
    "max_workers": max_workers,
}

base_lm_kwargs.update(language_model_params or {})
filtered_kwargs = {k: v for k, v in base_lm_kwargs.items() if v is not None}
config = factory.ModelConfig(
    model_id=model_id, provider_kwargs=filtered_kwargs
)

language_model = factory.create_model(
    config=config,
    examples=prompt_template.examples if use_schema_constraints else None,
    use_schema_constraints=use_schema_constraints,
    fence_output=fence_output,
)

fence_output = language_model.requires_fence_output
language_model

In [ ]:
# ===> Resolver

resolver_params = None

resolver_defaults = {
    "fence_output": fence_output,
    "format_type": format_type,
    "extraction_attributes_suffix": "_attributes",
    "extraction_index_suffix": None,
}
resolver_defaults.update(resolver_params or {})

res = resolver.Resolver(**resolver_defaults)
res

In [ ]:
# ===> Annotator
annotator = annotation.Annotator(
    language_model=language_model,
    prompt_template=prompt_template,
    format_type=format_type,
    fence_output=fence_output,
)

# Update prompt generator with custom settings
annotator._prompt_generator = prompting.QAPromptGenerator(
    prompt_template,
    format_type=data.FormatType.JSON,
    examples_heading="Ejemplos:\n--------\n",
    question_prefix="Input: ",
    answer_prefix="Output: ",
    fence_output=False,
)

In [ ]:
annotator._prompt_generator

In [ ]:
# Annotation run
# if isinstance(text_or_documents, str):
#     return annotator.annotate_text(
#         text=text_or_documents,
#         resolver=res,
#         max_char_buffer=max_char_buffer,
#         batch_length=batch_length,
#         additional_context=additional_context,
#         debug=debug,
#         extraction_passes=extraction_passes,
#         max_workers=max_workers,
#     )
# else:
#     documents = cast(Iterable[data.Document], text_or_documents)
#     return annotator.annotate_documents(
#         documents=documents,
#         resolver=res,
#         max_char_buffer=max_char_buffer,
#         batch_length=batch_length,
#         debug=debug,
#         extraction_passes=extraction_passes,
#         max_workers=max_workers,
#     )

In [ ]:
text = """
    JUZGADO DE FAMILIA N.º 1 DE LA CIUDAD DE SAN LORENZO
    Expediente N.º 3187/2023
    Carátula: Rodríguez, Ana Carolina c/ Fernández, Diego Esteban s/ Violencia Familiar
    SENTENCIA
    En la ciudad de San Lorenzo, Provincia de Santa Fe, a los 22 días del mes de noviembre de 2023, siendo las 09:15 horas, la Sra. Jueza de Familia Dra. Verónica Salvatierra dicta la presente resolución en los autos caratulados “Rodríguez, Ana Carolina c/ Fernández, Diego Esteban s/ Violencia Familiar”, Expte. N.º 3187/2023.
"""

In [ ]:
annotator.annotate_text(
    text,
    resolver=res,
    max_char_buffer=1024,
    **{"timeout": 600, "keep_alive": "10m"},
)

## `langextract.annotation.Annotator` review

### `annotate_text()`

In [ ]:
# start_time = time.time() if debug else None

# documents = [
#     data.Document(
#         text=text,
#         document_id=None,
#         additional_context=additional_context,
#     )
# ]

# TODO: zoom in on this function
# annotations = list(
#     self.annotate_documents(
#         documents,
#         resolver,
#         max_char_buffer,
#         batch_length,
#         debug,
#         extraction_passes,
#         **kwargs,
#     )
# )
# assert (
#     len(annotations) == 1
# ), f"Expected 1 annotation but got {len(annotations)} annotations."

# if debug and annotations[0].extractions:
#     elapsed_time = time.time() - start_time if start_time else None
#     num_extractions = len(annotations[0].extractions)
#     unique_classes = len(
#         set(e.extraction_class for e in annotations[0].extractions)
#     )
#     num_chunks = len(text) // max_char_buffer + (
#         1 if len(text) % max_char_buffer else 0
#     )

#     progress.print_extraction_summary(
#         num_extractions,
#         unique_classes,
#         elapsed_time=elapsed_time,
#         chars_processed=len(text),
#         num_chunks=num_chunks,
#     )

# return data.AnnotatedDocument(
#     document_id=annotations[0].document_id,
#     extractions=annotations[0].extractions,
#     text=annotations[0].text,
# )